# Q&A across documents with LangChain and LangSmith

## What this notebook does

This notebook builds a Retrieval-Augmented Generation (RAG) chatbot that can answer questions across multiple documents. It starts by loading source material from Wikipedia, DOCX, PDF, and TXT files, splitting that material into chunks, embedding the chunks, and storing them in a Chroma vector database.

After ingestion, the notebook shows how to query the vector store directly, then how to connect retrieval to a LangChain RAG chain so an LLM can synthesize answers from the retrieved context. The final section adds chat message history, allowing the chatbot to handle follow-up questions with conversational memory.

The notebook can run with OpenAI, Ollama, or Gemini. The provider and model settings are loaded from the project-root `.env` file when available.


In [ ]:
# Select the LLM provider for the notebook: "openai", "ollama", or "gemini".
# Values are loaded from the project-root .env file when present.
import os
import getpass
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


def load_project_env(filename=".env", override=True):
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        env_path = directory / filename
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if override or key not in os.environ:
                    os.environ[key] = value
            return env_path
    return None


PROJECT_ENV_PATH = load_project_env(override=True)

# LangSmith tracing is configured before any LangChain objects are created.
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "Q & A chatbot")
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

SUPPORTED_LLM_PROVIDERS = {"openai", "ollama", "gemini"}
if LLM_PROVIDER not in SUPPORTED_LLM_PROVIDERS:
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}. Use one of: {sorted(SUPPORTED_LLM_PROVIDERS)}")


def _get_api_key(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return getpass.getpass(f"Enter your {names[0]}")


class GeminiEmbeddingsOneByOne:
    def __init__(self, embeddings, delay_seconds=0.7, max_retries=5):
        self.embeddings = embeddings
        self.delay_seconds = delay_seconds  # pause between requests to stay under the rate limit
        self.max_retries = max_retries

    def _embed_one(self, text):
        import time
        for attempt in range(self.max_retries + 1):
            try:
                return self.embeddings.embed_documents([text], batch_size=1)[0]
            except Exception as exc:
                # Back off and retry only on quota errors (429 RESOURCE_EXHAUSTED)
                if "RESOURCE_EXHAUSTED" not in str(exc) or attempt == self.max_retries:
                    raise
                wait = 5 * 2 ** attempt
                print(f"Rate limit hit, retrying in {wait}s...")
                time.sleep(wait)

    def embed_documents(self, texts):
        # Gemini embedding models can return one vector per batch with this
        # LangChain version. Chroma expects one vector per document, so embed
        # each chunk separately, with a small delay between requests.
        import time
        vectors = []
        for text in texts:
            vectors.append(self._embed_one(text))
            time.sleep(self.delay_seconds)
        return vectors

    def embed_query(self, text):
        return self.embeddings.embed_query(text)


def ensure_ollama_model_available(model_name):
    import ollama

    try:
        installed_models = {model.model for model in ollama.list().models}
    except Exception as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {OLLAMA_BASE_URL}. "
            "Start Ollama before running this notebook."
        ) from exc

    model_aliases = {name.removesuffix(":latest") for name in installed_models}
    if model_name not in installed_models and model_name not in model_aliases:
        raise RuntimeError(
            f'Ollama model "{model_name}" is not installed. '
            f'Run `ollama pull {model_name}` in a terminal, then restart the notebook kernel.'
        )


def get_embeddings_model():
    if LLM_PROVIDER == "openai":
        return OpenAIEmbeddings(
            model=OPENAI_EMBEDDING_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_EMBEDDING_MODEL)
        return OllamaEmbeddings(
            model=OLLAMA_EMBEDDING_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        embeddings = GoogleGenerativeAIEmbeddings(
            model=GEMINI_EMBEDDING_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
        return GeminiEmbeddingsOneByOne(embeddings)


def get_chat_model():
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(
            model=OPENAI_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_MODEL)
        return ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        return ChatGoogleGenerativeAI(
            model=GEMINI_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )


def message_text(message):
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


print(f"Loaded env from: {PROJECT_ENV_PATH or 'not found'}")
print(f"Using {LLM_PROVIDER} chat model and matching embeddings")
if LLM_PROVIDER == "ollama":
    print(f"Ollama chat model: {OLLAMA_MODEL}")
elif LLM_PROVIDER == "gemini":
    print(f"Gemini chat model: {GEMINI_MODEL}")
print(f"LangSmith tracing: {os.getenv('LANGSMITH_TRACING', 'false')} | project: {os.getenv('LANGSMITH_PROJECT')}")


### 7.2 Vector store content ingestion

Before the system can answer questions across documents, the source material must be loaded into a vector database. This notebook ingests content about Paestum from several sources and formats, so the later retrieval step can search across a small knowledge base instead of a single text.

The loaders read the original files, the splitter breaks them into searchable chunks, the embedding model turns each chunk into a vector, and Chroma stores both the vectors and the original text for later retrieval.


In [17]:
from langchain_community.document_loaders import WikipediaLoader, Docx2txtLoader, PyPDFLoader, TextLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


## Setting up vector database and embeddings

### Splitting and storing documents

The splitter creates chunks of about 500 characters. Smaller chunks make retrieval more precise because a query can match a focused passage instead of a large mixed section. Here the overlap is set to `0` to keep the example simple.

The embedding model is selected through `get_embeddings_model()`, so the same notebook can build the vector store with OpenAI, Ollama, or Gemini embeddings. The Chroma collection named `tourist_info` stores the resulting chunks and their embeddings.


In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=0)
embeddings_model = get_embeddings_model()
vector_db = Chroma(
    collection_name="tourist_info",
    embedding_function=embeddings_model,
)


### Loading the first document sources

Each loader knows how to read one source type. `WikipediaLoader` retrieves web content, `Docx2txtLoader` reads Word files, `PyPDFLoader` reads PDFs, and `TextLoader` reads plain text. After loading, every document is split and added to the same Chroma collection.

The Wikipedia loader may bring in more material than expected because referenced Wikipedia pages can also be loaded. That is useful for coverage, but it also means retrieval results should be inspected to understand which source supplied the answer.


In [20]:
import wikipedia
wikipedia.set_user_agent("llm-book-notebook/1.0 (thimoty.barbieri@gmail.com)")


wikipedia_loader = WikipediaLoader(query="Paestum")
wikipedia_chunks = text_splitter.split_documents(
    wikipedia_loader.load())
vector_db.add_documents(wikipedia_chunks)

['b29f301b-4476-4e86-9704-aaa685fe249e',
 '47977581-2596-4c63-88d5-e2200c2c7bba',
 '5451cbab-10bd-43c7-ad13-0986f9e8398b',
 '8f539b15-20b3-43e4-ae11-86b2ad7955ef',
 'fc639b9a-fbbf-41d1-8b83-94b7867337a7',
 'db44b84b-5bbd-465a-b038-53f326967694',
 'a3a22819-3a95-46e8-9ad0-a5f9416c03bf',
 'b999653f-4739-4584-869f-65add6eec208',
 '6d94c3ed-6702-48de-8091-41a06cf4a4aa',
 '423b58f3-98f1-49f4-94e3-b6e1b529272c',
 'cbc890b3-a4d2-45ee-8465-4b644fc8131a',
 '4da9976f-d052-40da-8fdd-f826f79569b1',
 'c4a03dcf-cd9a-40df-94fc-6b6733848bca',
 '620ddfca-d0fb-43f8-b86c-5bbb02abc3a0',
 'b5207bd3-bcdd-47e3-84e8-486e0bccf60a',
 '39887a23-d3b3-4908-8862-1511afe301b4',
 '2cb153ce-a7e2-4e76-83cb-f2cbda139559',
 '04ce190d-bb52-4710-89b4-85d4398c5e6a',
 'ff81a84e-a5be-441e-9641-3e6811fc0fcb',
 '58623660-5c4f-4cac-8d59-f77613b5d457',
 '6fd49065-8fe1-46d8-8899-9bd552e9b1ce',
 'cf6b0a93-b1d8-4413-9c6e-8795486eeb55',
 '87cd7aae-580c-4f31-8403-dac7c5da543d',
 '3aee3a1a-6902-46e9-be16-851997485b40',
 '3d8d2be8-b42c-

In [21]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_chunks = text_splitter.split_documents(
    word_loader.load())
vector_db.add_documents(word_chunks)

['3cea568b-4307-41c6-89d7-f7afa654c252',
 '5a383d51-82b1-41b4-8d0a-cbce54288823',
 '1d58ca10-d054-4c6f-bc7e-88cb9d3ebe75',
 '618f02d7-3bbc-436a-bbca-867b62a1ab3b',
 '9d12b788-7523-44ee-b264-43f9653105ac',
 'b6d34f71-9b1c-4969-909a-05fbb62a5dd8',
 '72df779d-79ac-4a25-a58e-bbd59daf800e',
 '5c4c1305-6ca2-4c4f-a90f-45df143ed0d8']

In [6]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(
    pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['659f1b27-efca-45e9-93fe-ea2471b5999e',
 'd5b07620-aa71-4ada-bcce-4fee9e6922d4',
 'fe0ebdd8-a9e8-4f39-b9ab-0deef7fee879',
 '0238e7db-8caf-4ad2-acd8-073b0f9d6f0d',
 '6deb1980-7f0b-4239-943c-d35d230694b0',
 '641d8ae7-6abc-46c0-9b16-8042b51a5461',
 'b1a0e459-d1ea-49d9-81c1-03b85047d8ec',
 '44256428-9492-45f0-bed3-20515b0b8944',
 '90607849-b57b-45e2-b76f-d19d297b051c',
 '6bc09fdd-1bd5-4e54-9c10-ad581298700a',
 '1574fb31-ac13-4fe6-a374-cef3897fa619',
 'a2c1184a-4f34-461d-bb22-6a2e37ef5272',
 '7258ea54-b6db-471a-903c-1a43a2688df2',
 '5d548454-b833-4357-99e4-aac49ac66681',
 'b80d4acd-7d51-41c9-9e52-28a9576fbfbf',
 'cf436b4a-aff0-4231-bbe0-c66dcb743495',
 'ac363170-8615-4d71-a4d1-854319549f47',
 'f1c29e4a-d051-4534-9d5c-7b2b1723d141',
 '0fabc0ba-7dc7-48aa-8170-e51e912cc315',
 '8b714d5a-1db8-460c-8ec2-86b967b07deb',
 'b8c7a8fa-00b3-419a-851d-fff957dad66c',
 '711a224b-7e57-45bf-9343-5725f4cb6442',
 'c2ce9eb7-bb62-4636-9d98-0df91189b6d2',
 '61345f05-f668-4c71-b7e7-9bfd0ce0949a',
 '0c122d2a-5187-

In [7]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(
    txt_loader.load())
vector_db.add_documents(txt_chunks)

['5dd29725-28e8-44ba-bd6b-02cef21562ad']

### Removing duplication

### Why remove duplication

The ingestion pattern is the same for every loader: load documents, split them into chunks, and add the chunks to Chroma. Pulling that pattern into `split_and_import()` keeps the notebook easier to extend when additional file types or sources are added.


In [8]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)
     print(f"Ingested chunks created by {loader}")

In [9]:
wikipedia_loader = WikipediaLoader(query="Paestum")
split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Ingested chunks created by <langchain_community.document_loaders.wikipedia.WikipediaLoader object at 0x00000207DDCB66D0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x00000207DDD45110>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x00000207CA4E4490>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x00000207DC729110>


## Ingesting Multiple Documents from a Folder (two techniques)

### Ingesting many files from a folder

Loading one file at a time works for a few sources, but a real knowledge base usually contains many documents. The next cells show two approaches: iterate over the folder and choose the loader from the file extension, or use LangChain's `DirectoryLoader` when the optional unstructured dependencies are installed.


### 1) Iterating over all files in a folder

### Loader factory by extension

The loader factory maps file extensions to the proper LangChain loader. This keeps the folder loop generic: it does not need to know how to process DOCX, PDF, or TXT directly; it only asks `get_loader()` to instantiate the correct loader for each file.


In [10]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [11]:
import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename) #A Extract the file extension
    file_extension = file_extension.lstrip('.') #B Remove the leading dot from the extension
    
    loader_class = loader_classes.get(
        file_extension) #C Get the loader class from the dictionary
    
    if loader_class:
        return loader_class(filename) #D Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

#### Ingesting the files from the folder (Exercise solution)

### Folder ingestion loop

The loop walks through `CilentoTouristInfo`, skips directories, creates the appropriate loader for each file, and reuses `split_and_import()` to add the content to Chroma. Unsupported extensions are reported instead of stopping the whole ingestion process.


In [12]:
folder_path = "CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file
   
    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

Loader for Acciaroli.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x00000207DDCEDB10>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x00000207DDCEDB10>
Loader for Cape Palinuro.txt: <langchain_community.document_loaders.text.TextLoader object at 0x00000207DDCEFA10>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x00000207DDCEFA10>
Loader for Casalvelino.txt: <langchain_community.document_loaders.text.TextLoader object at 0x00000207DDCF0D90>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x00000207DDCF0D90>
Loader for Cilentan coast.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x00000207E07155D0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x00000207E07155D0>
Loader for Cilento Coast Map and Travel Guide.docx: <langchain_community.docum

### 2) Ingesting all files with with DirectoryLoader

### DirectoryLoader alternative

`DirectoryLoader` can ingest a whole folder with a glob pattern, but it depends on optional unstructured-document packages whose installation can vary by operating system. The explicit folder loop above is more transparent; this cell shows the shorter alternative for environments where those dependencies are available.


In [15]:
# ONLY RUN THIS IF YOU HAVE SUCCESFULLY INSTALLED unstructured or langchain-unstructured
# THE INSTALLATION IS OPERATIVE SYSTEM SPECIFIC
# follow LangChain instructions at https://python.langchain.com/v0.2/docs/integrations/providers/unstructured/ or 
# Unstructured instructions at https://docs.unstructured.io/welcome#quickstart-unstructured-open-source-library
folder_path = "CilentoTouristInfo"
pattern = "**/*.{docx,pdf,txt}" #A Pattern to match .docx, .pdf, and .txt files

directory_loader = DirectoryLoader(folder_path, pattern) #B Initialize the DirectoryLoader with the folder path and pattern
split_and_import(directory_loader)

NameError: name 'DirectoryLoader' is not defined

### 7.3 Q&A across stored documents

After ingestion, the vector store can be queried across all stored documents. This is the point where the notebook checks whether the chunking and ingestion strategy worked: a question should retrieve relevant passages even when the answer is spread across different files and sources.


## Querying the vector store directly

### Direct vector store query

The direct `similarity_search()` call asks Chroma for the four closest chunks. This does not generate a final answer; it only returns the retrieved documents and metadata. Inspecting these raw results is useful before adding the LLM because it shows what context the RAG chain will receive.


In [13]:
query = "Where was Poseidonia and who renamed it to Paestum?" 
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='eea99bf2-1ae1-4c0c-bec1-2811c02752ea', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'title': 'Paestum', 'summary': 'Paestum ( PEST-əm, US also  PEE-stəm, Latin: [ˈpae̯stũː]) was a major ancient Greek city on the coast of the Tyrrhenian Sea, in Magna Graecia. The ruins of Paestum are famous for their three ancient Greek temples in the Doric order dating from about 550 to 450 BCE that are in an excellent state of preservation. The city walls and amphitheatre are largely intact, and the bottom of the walls of many other structures remain, as well as paved roads. The site is open to the public, and there is a modern national museum within it, which also contains the finds from the associated Greek site of Foce del Sele.\nPaestum was established around 600 BCE by settlers from Sybaris, a Greek colony in southern Italy, under the name of Poseidonia (Ancient Greek: Ποσειδωνία). The city thrived as a Greek settlement for about two centuries, witnessing the develop

In [14]:
len(results)

4

## Asking a question through a RAG chain

### Prompting the RAG answer

The prompt template defines the contract for answer generation. It receives `{context}` from the retriever and `{question}` from the user, then instructs the model to answer only from the retrieved material and to say when it does not know.


In [16]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end. 
If you don't know the answer, just say that you don't know, 
don't try to make up an answer.
Use three sentences maximum and keep the 
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

### Completing the RAG chain

The RAG chain has three moving parts: the retriever supplies context, `RunnablePassthrough` forwards the original question, and the chat model generates the answer. LCEL connects these pieces with `|`, so the output of the prompt becomes the input to the selected LLM.


In [17]:
retriever = vector_db.as_retriever()

In [18]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

In [ ]:
chatbot = get_chat_model()


In [20]:
# set up RAG chain

rag_chain = {"context": retriever, 
             "question": question_feeder}|rag_prompt|chatbot

### Executing the chain

`execute_chain()` is a thin wrapper around `chain.invoke(question)`. Keeping it as a function makes later examples easier to compare, especially when message history is added in the next section.


In [21]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [22]:
question = """Where was Poseidonia and who renamed 
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(message_text(answer))

- Poseidonia was located along the Gulf of Taranto in southern Italy. 
- It was renamed Paestum by the Lucanians (before 400 BCE). 
- Source: Paestum-Britannica.docx (Britannica entry on Paestum).


In [23]:
print(answer)

content='- Poseidonia was located along the Gulf of Taranto in southern Italy. \n- It was renamed Paestum by the Lucanians (before 400 BCE). \n- Source: Paestum-Britannica.docx (Britannica entry on Paestum).' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1730, 'prompt_tokens': 1531, 'total_tokens': 3261, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1664, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CXXc1jjfRjBIebjlA9PoYPOMtbciD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--ff3aa984-36d5-4f31-ad8e-d4ac12bdc6d8-0' usage_metadata={'input_tokens': 1531, 'output_tokens': 1730, 'total_tokens': 3261, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 

### Stateless follow-up question

This follow-up question is intentionally vague. The chain built so far is stateless: it receives only the current question, retrieves context for that question, and calls the LLM. It does not remember that the previous question was about Poseidonia and the Romans, so pronouns such as "they" can be misinterpreted.


In [24]:
question = """And then, what they do? 
Tell me only if you know. 
Also tell me the source""" 
answer = execute_chain(rag_chain, question)
print(message_text(answer))

They are the two paths in Parmenides’ work: The Way of Truth and The Way of Opinion. The Way of Truth deals with true reality (being), while The Way of Opinion deals with appearances and beliefs based on sense experience. Source: CilentoTouristInfo\Parmenides.docx.


### 7.4 Chatbot memory of message history

A chatbot needs message history to handle follow-up questions. Without memory, each question is treated as an isolated request. With memory, the prompt can include previous human and AI messages, giving the model enough conversational context to resolve references such as "they" or "then".


## Chatbot memory of message history

### Memory-enabled RAG chain

The memory-enabled chain switches from a plain `PromptTemplate` to `ChatPromptTemplate.from_messages()`. This lets the prompt include a system instruction, previous chat messages, retrieved context, and the current human question as separate chat roles.

`ChatMessageHistory` stores the conversation. `RunnableLambda(get_messages)` injects the current message history into the prompt each time the chain runs, and `execute_chain_with_memory()` updates the history after each model response.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = get_chat_model()
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    "retrieved_context": retriever, 
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')                                      
    return answer


### Testing the memory-enabled chain

The first question populates the message history with a concrete topic. The second, vaguer follow-up can then use that history to understand what the user is referring to. This demonstrates the difference between a stateless RAG engine and a conversational RAG chatbot.


In [26]:
question = """Where was Poseidonia and who renamed 
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(message_text(answer))

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed \nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was a Greek settlement on the Tyrrhenian coast of southern Italy, at the Gulf of Taranto (Magna Graecia). It was renamed Paestum by the Romans after their conquest in 273 BCE; earlier, the Lucanians had renamed it Paistos. Source: https://en.wikipedia.org/wiki/Paestum', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1232, 'prompt_tokens': 1594, 'total_tokens': 2826, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1152, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CXXcQ4mXRBIXiQWYr6FJhGJGlnJtW', 'service_tier': 'default', 

In [27]:
question = """And then what did they do? 
Also tell me the source""" 
answer = execute_chain_with_memory(rag_chain, question)
print(message_text(answer))

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed \nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was a Greek settlement on the Tyrrhenian coast of southern Italy, at the Gulf of Taranto (Magna Graecia). It was renamed Paestum by the Romans after their conquest in 273 BCE; earlier, the Lucanians had renamed it Paistos. Source: https://en.wikipedia.org/wiki/Paestum', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1232, 'prompt_tokens': 1594, 'total_tokens': 2826, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1152, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CXXcQ4mXRBIXiQWYr6FJhGJGlnJtW', 'service_tier': 'default', 

## Tracing with LangSmith

LangSmith tracing can be configured directly from the project-root `.env` file. The first notebook cell loads those values before any LangChain objects are created, so traces are captured as soon as you run the RAG chain cells.

Add your LangSmith key to `.env`:

```bash
LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```

The notebook sets these defaults automatically when they are not already defined:

```bash
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_PROJECT=Q & A chatbot
```

If `LANGSMITH_API_KEY` is present, the notebook enables tracing with:

```bash
LANGSMITH_TRACING=true
```

After editing `.env`, restart the kernel and run the notebook from the first cell so the environment is loaded before the LangChain chains are built. The RAG chain and memory-enabled RAG chain executions will then appear in the configured LangSmith project.
